# 🚗 Treinamento do YOLO de Caracteres para Placas Brasileiras (Mercosul & Antigas)

Este notebook treina um modelo **YOLOv8** especializado no reconhecimento de caracteres de placas veiculares brasileiras utilizando datasets do **Roboflow Universe**.

### 📌 Passos do Treinamento:
1. Configurar GPU gratuita no Colab (Ambiente de Execução > Alterar tipo de ambiente > GPU T4)
2. Instalar Ultralytics e Roboflow
3. Baixar o Dataset de Caracteres Mercosul do Roboflow
4. Treinar o modelo YOLOv8 para 36 classes (A-Z, 0-9)
5. Avaliar a precisão (mAP50) e exportar o arquivo `detector_caracteres.pt`

In [ ]:
# 1. Verificação da GPU e Instalação de Dependências
!nvidia-smi
!pip install -q ultralytics roboflow

## 📦 2. Download do Dataset via Roboflow Universe
Insira sua chave de API do Roboflow ou utilize o dataset público de caracteres brasileiros:

In [ ]:
from roboflow import Roboflow
import os

# Insira sua chave de API do Roboflow abaixo (ou utilize um dataset público):
ROBOFLOW_API_KEY = "SUA_API_KEY_AQUI"  # Obtenha gratuitamente em roboflow.com
WORKSPACE = "placas-brasil"
PROJECT = "caracteres-mercosul"
VERSION = 1

try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    dataset = project.version(VERSION).download("yolov8")
    data_yaml = os.path.join(dataset.location, "data.yaml")
    print(f"✅ Dataset pronto em: {data_yaml}")
except Exception as e:
    print(f"⚠️ Se não tiver chave de API, você pode fazer upload do zip do seu dataset diretamente no menu lateral do Colab e descompactar aqui!")
    print(e)

## 🚀 3. Treinamento do YOLOv8 de Caracteres
Treinamento da rede neural convolucional para identificação visual direta das 36 classes de letras e números.

In [ ]:
from ultralytics import YOLO

# Carrega o modelo base nano pré-treinado
model = YOLO("yolov8n.pt")

# Executa o treinamento com Data Augmentation
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=320,
    batch=16,
    patience=15,
    save=True,
    project="treino_caracteres_mercosul",
    name="yolo_caracteres",
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    degrees=5.0,
    scale=0.2,
    perspective=0.0005,
    fliplr=0.0,  # Não espelha caracteres horizontalmente
    mosaic=0.5
)

## 📊 4. Avaliação e Gráficos de Desempenho

In [ ]:
from IPython.display import Image, display
import glob

# Exibe matriz de confusão e resultados das métricas de precisão
confusion_matrix = glob.glob("treino_caracteres_mercosul/yolo_caracteres/confusion_matrix.png")
if confusion_matrix:
    display(Image(filename=confusion_matrix[0]))

results_png = glob.glob("treino_caracteres_mercosul/yolo_caracteres/results.png")
if results_png:
    display(Image(filename=results_png[0]))

## 💾 5. Baixar o Modelo Treinado (`detector_caracteres.pt`)
Execute a célula abaixo para baixar os pesos treinados e salvá-los na pasta `models/detector_caracteres.pt` do seu projeto local.

In [ ]:
from google.colab import files
import shutil

melhor_peso = "treino_caracteres_mercosul/yolo_caracteres/weights/best.pt"
destino = "detector_caracteres.pt"
shutil.copy(melhor_peso, destino)

print("🎉 Baixando o modelo para o seu computador...")
files.download(destino)